In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 범용 목적 지향적 의미 분할 시스템 - EXAONE 4.0.1
# ═══════════════════════════════════════════════════════════════════════
import os
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import re
import json
import time
import logging
from datetime import datetime
from typing import List, Dict, Tuple

# ═══════════════════════════════════════════════════════════════════════
# 1. 로깅 설정
# ═══════════════════════════════════════════════════════════════════════
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("semantic_chunking_analysis.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ═══════════════════════════════════════════════════════════════════════
# 2. 범용 프롬프트 (대상만 변경 가능)
# ═══════════════════════════════════════════════════════════════════════
SEMANTIC_CHUNKING_PROMPT = """다음 뉴스 텍스트에서 **{target}**을 언급한 부분을 의미론적으로 추출하세요.

**뉴스 텍스트:**
{text}

**의미 분할 원칙:**
1. **같은 주제/관점은 하나로 묶기**: 같은 내용을 다루는 여러 문장은 하나의 청크로 통합
2. **주제 전환 시에만 분할**: 새로운 논점이나 관점이 등장할 때만 새 청크 생성
3. **원문 그대로**: 텍스트 수정 없이 그대로 인용

**나쁜 예 vs 좋은 예:**
❌ 나쁜 분할 (문장별):
- 청크1: "첫 번째 문장"
- 청크2: "두 번째 문장"
- 청크3: "세 번째 문장"

✅ 좋은 분할 (주제별):
- 청크1: "첫 번째 문장 + 두 번째 문장 + 세 번째 문장" (같은 주제)

**출력 형식:**
{{
    "negative_chunks": [
        {{
            "chunk": "같은 주제의 문장들을 포함한 원문",
            "reason": "이 청크가 다루는 주제/관점 설명",
            "keywords": ["키워드1", "키워드2", "키워드3"]
        }}
    ],
    "has_negative_mention": true/false
}}

**중요:** 같은 주제는 반드시 하나로 묶으세요. 새로운 관점이 나올 때만 분할하세요.

**주의사항:**
1. 인공지능의 리스크(risk)와 관련된 언급만 추출하세요 (위험성, 문제점, 우려, 비판 등)
2. 문맥상 중립적이거나 긍정적인 내용은 제외하세요
3. 추출된 chunk는 원문 그대로 인용하세요
4. JSON 응답 시 탭(\\t), 캐리지 리턴(\\r), 백스페이스(\\b) 등의 제어문자를 절대 사용하지 마세요
5. 들여쓰기는 공백 4개를 사용하세요
6. JSON 문법을 정확히 지켜주세요 (쉼표, 따옴표 위치 등)

JSON만 출력:"""

# ═══════════════════════════════════════════════════════════════════════
# 3. 유틸리티 함수
# ═══════════════════════════════════════════════════════════════════════
def clean_text(text):
    """텍스트 정제"""
    if pd.isna(text) or text == "":
        return ""

    text = str(text)
    text = text.replace('\n', ' ')
    text = text.replace('\r', ' ')
    text = text.replace('\t', ' ')
    text = re.sub(r'[\x00-\x1f\x7f-\x9f]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def parse_json(response):
    """JSON 파싱"""
    try:
        # 제어문자 제거
        response = clean_text(response)

        # 마크다운 코드 블록 제거
        response = re.sub(r'```json\s*', '', response)
        response = re.sub(r'```\s*', '', response)

        # JSON 추출
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            result = json.loads(match.group(0))

            # 검증
            if "negative_chunks" not in result:
                result["negative_chunks"] = []
            if "has_negative_mention" not in result:
                result["has_negative_mention"] = len(result["negative_chunks"]) > 0

            return result

        return {"negative_chunks": [], "has_negative_mention": False, "parse_error": True}

    except json.JSONDecodeError as e:
        logger.warning(f"JSON 파싱 실패: {e}")
        return {"negative_chunks": [], "has_negative_mention": False, "parse_error": True}
    except Exception as e:
        logger.error(f"예외 발생: {e}")
        return {"negative_chunks": [], "has_negative_mention": False, "parse_error": True}

# ═══════════════════════════════════════════════════════════════════════
# 4. 의미 분할 분석기
# ═══════════════════════════════════════════════════════════════════════
class SemanticChunker:
    """EXAONE 4.0.1 기반 범용 의미 분할"""

    def __init__(self, api_base="http://218.156.182.231:1234/v1",
                 model_name="lgai-exaone_exaone-4.0.1-32b",
                 target_topic="AI의 위험성이나 리스크"):
        """
        범용 의미 분할 분석기 초기화

        Parameters:
            api_base: API 엔드포인트
            model_name: 모델명
            target_topic: 추출할 대상 (예: "AI의 위험성", "대통령 부정적 언급")
        """
        self.client = OpenAI(base_url=api_base, api_key="not-needed")
        self.model_name = model_name
        self.target_topic = target_topic

        logger.info(f"🚀 범용 의미 분할 분석기 초기화")
        logger.info(f"API: {api_base}")
        logger.info(f"모델: {model_name}")
        logger.info(f"🎯 추출 대상: {self.target_topic}")

        # 연결 테스트
        self.test_connection()

    def test_connection(self):
        """연결 테스트"""
        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": "안녕하세요"}],
                max_tokens=10
            )
            logger.info("✅ 연결 성공")
        except Exception as e:
            logger.error(f"❌ 연결 실패: {e}")
            raise

    def analyze_single_text(self, text: str, max_retries: int = 2) -> Dict:
        """단일 텍스트 의미 분할"""

        # 동적으로 target 삽입
        prompt = SEMANTIC_CHUNKING_PROMPT.format(
            target=self.target_topic,
            text=text
        )

        for attempt in range(max_retries):
            try:
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[
                        {
                            "role": "system",
                            "content": "당신은 인공지능의 위험성을 연구하는 컴퓨터공학자입니다. JSON 형식으로만 응답하세요."
                        },
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.0,
                    max_tokens=8196,
                    extra_body={
                        "chat_template_kwargs": {
                            "enable_thinking": False
                        }
                    }
                )

                response_text = response.choices[0].message.content.strip()
                result = parse_json(response_text)

                if not result.get("parse_error", False):
                    return result
                else:
                    logger.warning(f"시도 {attempt + 1}/{max_retries}: 파싱 실패")
                    if attempt < max_retries - 1:
                        time.sleep(1)

            except Exception as e:
                logger.error(f"시도 {attempt + 1}/{max_retries}: API 오류 - {e}")
                if attempt < max_retries - 1:
                    time.sleep(2)

        # 모든 시도 실패
        return {
            "negative_chunks": [],
            "has_negative_mention": False,
            "parse_error": True
        }

    def process_csv_with_checkpoint(self, input_file: str, output_file: str,
                                   text_column: str = 'text',
                                   checkpoint_interval: int = 10):
        """체크포인트 기반 CSV 처리"""

        logger.info(f"📚 CSV 파일 로드: {input_file}")
        df = pd.read_csv(input_file, encoding='utf-8-sig')

        if text_column not in df.columns:
            raise ValueError(f"❌ '{text_column}' 컬럼이 없습니다")

        # 체크포인트 경로
        base_name = os.path.splitext(output_file)[0]
        checkpoint_path = f"{base_name}_checkpoint.csv"
        index_file_path = f"{base_name}_processed_indices.txt"

        # 체크포인트 로드
        processed_indices = self.load_processed_indices(index_file_path)
        all_results = self.load_checkpoint_results(checkpoint_path)

        processed_set = set(processed_indices)

        logger.info(f"📊 총 {len(df)}개 행 (이미 처리: {len(processed_set)}개)")
        logger.info(f"💾 체크포인트: {checkpoint_path}")

        # 진행 상황
        pbar = tqdm(df.iterrows(), total=len(df), desc="🔍 의미 분할 중")

        for idx, row in pbar:
            # 이미 처리된 행 스킵
            if idx in processed_set:
                continue

            text = row[text_column]

            # 빈 텍스트 스킵
            if pd.isna(text) or text.strip() == "":
                processed_indices.append(idx)
                continue

            # 의미 분할 실행
            result = self.analyze_single_text(text)

            # 결과 저장 (언급이 있는 경우만)
            if not result.get("parse_error", False) and result.get("has_negative_mention", False):
                for chunk_info in result.get("negative_chunks", []):
                    # 원본 데이터 + 분할 결과
                    result_row = row.to_dict()
                    result_row['original_index'] = idx

                    # 키워드 처리
                    keywords = chunk_info.get("keywords", [])
                    if isinstance(keywords, list):
                        keywords_str = ", ".join(keywords)
                    else:
                        keywords_str = str(keywords)

                    result_row.update({
                        "chunk": chunk_info.get("chunk", ""),
                        "reason": chunk_info.get("reason", ""),
                        "keywords": keywords_str,
                        "processed_at": datetime.now().isoformat()
                    })

                    all_results.append(result_row)

            processed_indices.append(idx)

            # 주기적 체크포인트 저장
            if len(processed_indices) % checkpoint_interval == 0:
                self.save_checkpoint(checkpoint_path, all_results, list(df.columns))
                self.save_processed_indices(index_file_path, processed_indices)

            # 진행률 업데이트
            pbar.set_postfix({
                'processed': len(processed_indices),
                'chunks': len(all_results)
            })

            time.sleep(0.1)  # API 부하 방지

        pbar.close()

        # 최종 결과 저장
        logger.info(f"\n{'='*60}")
        logger.info(f"🎉 의미 분할 완료!")
        logger.info(f"{'='*60}")
        logger.info(f"📊 총 처리: {len(df)}개")
        logger.info(f"📝 추출된 청크: {len(all_results)}개")

        if all_results:
            result_df = pd.DataFrame(all_results)

            # 컬럼 순서 정리
            analysis_columns = ["chunk", "reason", "keywords", "processed_at", "original_index"]
            original_columns = [col for col in df.columns if col in result_df.columns]
            ordered_columns = original_columns + [col for col in analysis_columns if col in result_df.columns]

            result_df = result_df[ordered_columns]
            result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
            logger.info(f"💾 결과 저장: {output_file}")
        else:
            logger.warning(f"⚠️ '{self.target_topic}' 관련 언급이 발견되지 않았습니다")
            empty_df = pd.DataFrame(columns=list(df.columns) + ["chunk", "reason", "keywords", "processed_at"])
            empty_df.to_csv(output_file, index=False, encoding='utf-8-sig')

        # 최종 체크포인트 저장
        self.save_checkpoint(checkpoint_path, all_results, list(df.columns))
        self.save_processed_indices(index_file_path, processed_indices)

        logger.info(f"{'='*60}\n")

        return all_results

    # ═══════════════════════════════════════════════════════════════════
    # 체크포인트 관련 함수
    # ═══════════════════════════════════════════════════════════════════
    def save_checkpoint(self, checkpoint_path: str, results: List[Dict], columns: List[str]):
        """체크포인트 저장"""
        if results:
            result_df = pd.DataFrame(results)
            analysis_cols = ["chunk", "reason", "keywords", "processed_at", "original_index"]
            all_cols = columns + [col for col in analysis_cols if col not in columns]
            existing_cols = [col for col in all_cols if col in result_df.columns]
            result_df = result_df[existing_cols]
            result_df.to_csv(checkpoint_path, index=False, encoding='utf-8-sig')
        else:
            empty_df = pd.DataFrame(columns=columns + ["chunk", "reason", "keywords", "processed_at"])
            empty_df.to_csv(checkpoint_path, index=False, encoding='utf-8-sig')

    def load_checkpoint_results(self, checkpoint_path: str) -> List[Dict]:
        """체크포인트 결과 로드"""
        if os.path.exists(checkpoint_path):
            try:
                df = pd.read_csv(checkpoint_path, encoding='utf-8-sig')
                if len(df) > 0:
                    logger.info(f"📂 체크포인트 로드: {len(df)}개 행")
                    return df.to_dict('records')
            except Exception as e:
                logger.error(f"체크포인트 로드 실패: {e}")
        return []

    def save_processed_indices(self, index_file_path: str, indices: List[int]):
        """처리된 인덱스 저장"""
        with open(index_file_path, 'w') as f:
            for idx in indices:
                f.write(f"{idx}\n")

    def load_processed_indices(self, index_file_path: str) -> List[int]:
        """처리된 인덱스 로드"""
        if os.path.exists(index_file_path):
            with open(index_file_path, 'r') as f:
                return [int(line.strip()) for line in f if line.strip()]
        return []

# ═══════════════════════════════════════════════════════════════════════
# 5. 메인 함수
# ═══════════════════════════════════════════════════════════════════════
def main():
    logger.info("="*60)
    logger.info("🚀 범용 의미 분할 시스템 시작")
    logger.info("="*60 + "\n")

    # ═══════════════════════════════════════════════════════════════════
    # 설정값 (여기만 수정하면 됨!)
    # ═══════════════════════════════════════════════════════════════════

    # 추출 대상 설정 (이것만 바꾸면 다른 주제에도 사용 가능!)
    TARGET_TOPIC = "인공지능(AI, LLM, VLM, AGI, ASI 등)의 리스크(risk)나 위험성"
    # TARGET_TOPIC = "대통령에 대한 부정적 언급"
    # TARGET_TOPIC = "기업의 비리나 스캔들"
    # TARGET_TOPIC = "환경 오염이나 기후 위기"

    # 파일 경로
    INPUT_FILE = "/mnt/c/Users/com/Desktop/20250625+청크테스트_001.csv"
    OUTPUT_FILE = "/mnt/c/Users/com/Desktop/20250625+청크테스트_001++.csv"

    # 기타 설정
    TEXT_COLUMN = "text"  # 분석할 텍스트 컬럼명
    API_BASE = "http://218.156.182.231:1234/v1"
    MODEL_NAME = "lgai-exaone_exaone-4.0.1-32b"
    CHECKPOINT_INTERVAL = 10  # 10개마다 체크포인트 저장

    # ═══════════════════════════════════════════════════════════════════

    # 경로 검증
    if not os.path.exists(INPUT_FILE):
        logger.error(f"❌ 입력 파일 없음: {INPUT_FILE}")
        return None

    # 출력 디렉토리 생성
    output_dir = os.path.dirname(OUTPUT_FILE)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
        logger.info(f"📁 출력 디렉토리 생성: {output_dir}")

    # 타임스탬프 추가
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name, ext = os.path.splitext(OUTPUT_FILE)
    output_path = f"{base_name}_{timestamp}{ext}"

    logger.info(f"🎯 추출 대상: {TARGET_TOPIC}")
    logger.info(f"📄 입력: {INPUT_FILE}")
    logger.info(f"📄 출력: {output_path}")
    logger.info(f"🔧 API: {API_BASE}")
    logger.info(f"🤖 모델: {MODEL_NAME}\n")

    try:
        # 분석 실행
        chunker = SemanticChunker(
            api_base=API_BASE,
            model_name=MODEL_NAME,
            target_topic=TARGET_TOPIC  # ← 대상 전달
        )

        results = chunker.process_csv_with_checkpoint(
            input_file=INPUT_FILE,
            output_file=output_path,
            text_column=TEXT_COLUMN,
            checkpoint_interval=CHECKPOINT_INTERVAL
        )

        logger.info("🎊 모든 작업 완료!")
        return results

    except Exception as e:
        logger.error(f"❌ 처리 중 오류: {e}")
        raise

if __name__ == "__main__":
    main()

In [ ]:
{
  "type": "object",
  "properties": {
    "negative_chunks": {
      "type": "array",
      "description": "AI 리스크 언급 청크 배열",
      "items": {
        "type": "object",
        "properties": {
          "chunk": {
            "type": "string",
            "description": "원문 그대로 인용된 AI 리스크 구문"
          },
          "reason": {
            "type": "string",
            "description": "해당 리스크 범주 설명"
          },
          "keywords": {
            "type": "array",
            "description": "핵심 키워드 3-5개",
            "items": {
              "type": "string"
            },
            "minItems": 1,
            "maxItems": 5
          }
        },
        "required": [
          "chunk",
          "reason",
          "keywords"
        ]
      }
    },
    "has_negative_mention": {
      "type": "boolean",
      "description": "AI 리스크 언급 존재 여부"
    }
  },
  "required": [
    "negative_chunks",
    "has_negative_mention"
  ]
}